In [ ]:
"""
Merge Glasser+Tian + Nettekoven + STN spheres into a single 448-ROI NIfTI atlas.

Inputs (all in /mnt/movement/users/jaizor/xtra/derivatives/all_atlas):
    glasser_360_MNI152NLin6Asym.nii.gz      - Glasser cortical parcellation (360 ROIs)
    tian_subcortex_54_MNI152NLin6Asym.nii    - Tian subcortical parcellation (54 ROIs)
    atl-NettekovenSym32_space-MNI_dseg.nii   - Nettekoven cerebellar parcellation (32 ROIs)
    
Output:
    CIMT_448ROIs_atlas.nii.gz               - Unified 448-ROI atlas
    cimt_atlas_labels.csv                   - Copied alongside for reference

NIfTI label → CSV index mapping:
    Labels 1-360   → CSV indices 0-359   (Glasser)
    Labels 361-414 → CSV indices 360-413 (Tian)
    Labels 415-446 → CSV indices 414-445 (Nettekoven)
    Labels 447-448 → CSV indices 446-447 (STN)
"""

import nibabel as nib
import numpy as np
from pathlib import Path
from nilearn import image
import shutil

# =============================================================================
# CONFIGURATION
# =============================================================================
ATLAS_DIR = Path("/mnt/movement/users/jaizor/xtra/derivatives/all_atlas")
OUTPUT_FILE = ATLAS_DIR / "CIMT_448ROIs_atlas.nii.gz"

# STN coordinates (MNI mm) — from your MEG pipeline
STN_COORDS = {
    447: [-11.89, -14.51, -6.40],  # Left STN  → CSV index 446
    448: [12.53, -13.97, -6.57]    # Right STN → CSV index 447
}
STN_RADIUS_MM = 5.0

# =============================================================================
# 1. LOAD REFERENCE SPACE (Glasser as template)
# =============================================================================
print("=" * 60)
print("CIMT 448-ROI Atlas Merge")
print("=" * 60)

print("\n[1/5] Loading reference space from Glasser atlas...")
glasser_img = nib.load(ATLAS_DIR / "glasser_360_MNI152NLin6Asym.nii.gz")
ref_affine = glasser_img.affine
ref_shape = glasser_img.shape
voxel_size = np.abs(np.diag(ref_affine)[:3])
print(f"      Shape: {ref_shape}")
print(f"      Voxel size: {voxel_size} mm")

# =============================================================================
# 2. GLASSER + TIAN (labels 1-414)
# =============================================================================
print("\n[2/5] Processing Glasser (1-360) + Tian (361-414)...")

# Glasser: keep as-is (labels 1-360)
glasser_data = glasser_img.get_fdata().astype(np.int32)
print(f"      Glasser unique labels: {len(np.unique(glasser_data)) - 1}")

# Tian: resample to Glasser grid, offset by +360 (labels 361-414)
tian_img = nib.load(ATLAS_DIR / "tian_subcortex_54_MNI152NLin6Asym.nii")
tian_resampled = image.resample_to_img(tian_img, glasser_img, interpolation='nearest')
tian_data = tian_resampled.get_fdata().astype(np.int32)
tian_data[tian_data > 0] += 360
print(f"      Tian unique labels (after offset): {len(np.unique(tian_data)) - 1}")

# Merge: Tian takes precedence where both have labels (shouldn't overlap)
gt_data = np.where(tian_data > 0, tian_data, glasser_data)
gt_rois = len(np.unique(gt_data)) - 1
print(f"      Combined Glasser+Tian: {gt_rois} ROIs (expected 414)")

if gt_rois != 414:
    print(f"      ⚠️  WARNING: Expected 414 ROIs, got {gt_rois}")

# =============================================================================
# 3. NETTEKOVEN CEREBELLUM (labels 415-446)
# =============================================================================
print("\n[3/5] Processing Nettekoven cerebellum (415-446)...")

nk_img = nib.load(ATLAS_DIR / "atl-NettekovenSym32_space-MNI_dseg.nii")
nk_resampled = image.resample_to_img(nk_img, glasser_img, interpolation='nearest')
nk_data = nk_resampled.get_fdata().astype(np.int32)
nk_data[nk_data > 0] += 414  # Nettekoven 1-32 → 415-446

nk_rois = len(np.unique(nk_data)) - 1
print(f"      Nettekoven ROIs: {nk_rois} (expected 32)")

if nk_rois != 32:
    print(f"      ⚠️  WARNING: Expected 32 ROIs, got {nk_rois}")

# =============================================================================
# 4. STN SPHERES (labels 447-448)
# =============================================================================
print("\n[4/5] Generating STN spheres (447-448)...")

stn_data = np.zeros(ref_shape, dtype=np.int32)

for label, mni_coord in STN_COORDS.items():
    # Convert MNI coordinates to voxel indices
    vox = np.round(
        nib.affines.apply_affine(np.linalg.inv(ref_affine), mni_coord)
    ).astype(int)
    
    # Check bounds
    if np.any(vox < 0) or np.any(vox >= ref_shape):
        print(f"      ⚠️  STN coordinate {mni_coord} maps to voxel {vox} — OUT OF BOUNDS")
        continue
    
    # Bounding box around sphere center
    radius_vox = int(np.ceil(STN_RADIUS_MM / voxel_size[0])) + 1
    
    x_min, x_max = max(0, vox[0] - radius_vox), min(ref_shape[0], vox[0] + radius_vox + 1)
    y_min, y_max = max(0, vox[1] - radius_vox), min(ref_shape[1], vox[1] + radius_vox + 1)
    z_min, z_max = max(0, vox[2] - radius_vox), min(ref_shape[2], vox[2] + radius_vox + 1)
    
    # Create distance grid within bounding box
    xx, yy, zz = np.mgrid[x_min:x_max, y_min:y_max, z_min:z_max]
    distances = np.sqrt(
        ((xx - vox[0]) * voxel_size[0]) ** 2 +
        ((yy - vox[1]) * voxel_size[1]) ** 2 +
        ((zz - vox[2]) * voxel_size[2]) ** 2
    )
    
    sphere_mask = distances <= STN_RADIUS_MM
    n_vox = np.sum(sphere_mask)
    stn_data[x_min:x_max, y_min:y_max, z_min:z_max][sphere_mask] = label
    
    side = "Left" if label == 447 else "Right"
    print(f"      {side} STN: label={label}, center={mni_coord}, {n_vox} voxels")

# =============================================================================
# 5. MERGE & VALIDATE
# =============================================================================
print("\n[5/5] Merging and validating...")

# Check spatial overlaps before merging
overlap_gt_nk = np.sum((gt_data > 0) & (nk_data > 0))
overlap_gt_stn = np.sum((gt_data > 0) & (stn_data > 0))
overlap_nk_stn = np.sum((nk_data > 0) & (stn_data > 0))

print(f"      Overlap GT-Nettekoven:  {overlap_gt_nk} voxels")
print(f"      Overlap GT-STN:         {overlap_gt_stn} voxels")
print(f"      Overlap Nettekoven-STN: {overlap_nk_stn} voxels")

if overlap_gt_nk > 0:
    print(f"      ⚠️  WARNING: Cortex/subcortex overlaps with cerebellum!")
if overlap_gt_stn > 0:
    print(f"      ⚠️  WARNING: STN overlaps with existing labels!")
if overlap_nk_stn > 0:
    print(f"      ⚠️  WARNING: STN overlaps with cerebellum!")

# Combine (later atlases overwrite earlier ones in case of overlap)
combined = gt_data.copy()
combined[nk_data > 0] = nk_data[nk_data > 0]
combined[stn_data > 0] = stn_data[stn_data > 0]

# Validate complete label set
unique_labels = set(np.unique(combined)) - {0}
n_rois = len(unique_labels)
expected = set(range(1, 449))
missing = sorted(expected - unique_labels)
extra = sorted(unique_labels - expected)

print(f"\n      Total ROIs in output: {n_rois} / 448")

if missing:
    print(f"      ❌ MISSING: {missing}")
else:
    print(f"      ✅ No missing ROIs")

if extra:
    print(f"      ❌ EXTRA (unexpected labels): {extra}")
else:
    print(f"      ✅ No extra labels")

# =============================================================================
# 6. SAVE
# =============================================================================
print(f"\n{'=' * 60}")
print("Saving...")

combined_img = nib.Nifti1Image(combined.astype(np.int32), ref_affine, glasser_img.header)
combined_img.set_data_dtype(np.int32)
combined_img.to_filename(OUTPUT_FILE)

# Also copy the labels CSV to the same directory if not already there
csv_src = ATLAS_DIR / "cimt_atlas_labels.csv"
if csv_src.exists():
    print(f"      Labels CSV already present: {csv_src.name}")
else:
    print(f"      ⚠️  Labels CSV not found in atlas directory")

print(f"      Atlas saved: {OUTPUT_FILE}")
print(f"      Shape: {combined.shape}")
print(f"      Data type: {combined.dtype}")
print(f"\n{'=' * 60}")
print("Done.")
print(f"{'=' * 60}")

CIMT 448-ROI Atlas Merge

[1/5] Loading reference space from Glasser atlas...
      Shape: (182, 218, 182)
      Voxel size: [1. 1. 1.] mm

[2/5] Processing Glasser (1-360) + Tian (361-414)...
      Glasser unique labels: 360
      Tian unique labels (after offset): 54
      Combined Glasser+Tian: 414 ROIs (expected 414)

[3/5] Processing Nettekoven cerebellum (415-446)...
      Nettekoven ROIs: 32 (expected 32)

[4/5] Generating STN spheres (447-448)...
      Left STN: label=447, center=[-11.89, -14.51, -6.4], 515 voxels
      Right STN: label=448, center=[12.53, -13.97, -6.57], 515 voxels

[5/5] Merging and validating...
      Overlap GT-Nettekoven:  499 voxels
      Overlap GT-STN:         22 voxels
      Overlap Nettekoven-STN: 0 voxels
      ⚠️  WARNING: Cortex/subcortex overlaps with cerebellum!
      ⚠️  WARNING: STN overlaps with existing labels!

      Total ROIs in output: 448 / 448
      ✅ No missing ROIs
      ✅ No extra labels

Saving...
      Labels CSV already present: c